In [3]:
# Retention / Cohorts / Funnels
# Build a monthly cohort retention table (signup month × months since signup).
# Day-1 / Day-7 / Day-30 retention for new users.
# Funnel analysis: of users who viewed a page, how many added to cart, how many purchased?

In [4]:
from pyspark.sql.functions import col, to_date

users_data = [
    (1, "Alice", "2026-01-05"),
    (2, "Bob", "2026-01-10"),
    (3, "Charlie", "2026-01-20"),
    (4, "David", "2026-02-03"),
    (5, "Eva", "2026-02-15"),
    (6, "Frank", "2026-03-01")
]

users_columns = ["user_id", "user_name", "signup_date"]

users_df = spark.createDataFrame(users_data, users_columns)

users_df = users_df.withColumn(
    "signup_date",
    to_date(col("signup_date"))
)

activity_data = [
    (1, "2026-01-06"),
    (1, "2026-02-10"),
    (1, "2026-03-05"),

    (2, "2026-01-15"),
    (2, "2026-02-20"),

    (3, "2026-01-25"),

    (4, "2026-02-05"),
    (4, "2026-03-10"),

    (5, "2026-02-20"),

    (6, "2026-03-05"),
    (6, "2026-04-01")
]

activity_columns = ["user_id", "activity_date"]

activity_df = spark.createDataFrame(activity_data, activity_columns)

activity_df = activity_df.withColumn(
    "activity_date",
    to_date(col("activity_date"))
)

users_df.createOrReplaceTempView("users")
activity_df.createOrReplaceTempView("user_activity")

NameError: name 'spark' is not defined

In [17]:
#Funnel analysis: of users who viewed a page, how many added to cart, how many purchased?

In [18]:
from pyspark.sql.functions import col, to_timestamp

events_data = [
    (1, "view_page",   "2026-01-01 09:00:00"),
    (1, "add_to_cart", "2026-01-01 09:05:00"),
    (1, "purchase",   "2026-01-01 09:10:00"),

    (2, "view_page",   "2026-01-01 10:00:00"),
    (2, "add_to_cart", "2026-01-01 10:10:00"),

    (3, "view_page",   "2026-01-01 11:00:00"),

    (4, "view_page",   "2026-01-02 12:00:00"),
    (4, "purchase",    "2026-01-02 12:20:00"),  # purchased but no add_to_cart

    (5, "add_to_cart", "2026-01-02 13:00:00"),  # no view_page

    (6, "view_page",   "2026-01-03 14:00:00"),
    (6, "add_to_cart", "2026-01-03 14:05:00"),
    (6, "purchase",    "2026-01-03 14:15:00")
]

columns = ["user_id", "event_name", "event_time"]

events_df = spark.createDataFrame(events_data, columns)

events_df = events_df.withColumn(
    "event_time",
    to_timestamp(col("event_time"))
)

events_df.createOrReplaceTempView("events")

events_df.show(truncate=False)

+-------+-----------+-------------------+
|user_id|event_name |event_time         |
+-------+-----------+-------------------+
|1      |view_page  |2026-01-01 09:00:00|
|1      |add_to_cart|2026-01-01 09:05:00|
|1      |purchase   |2026-01-01 09:10:00|
|2      |view_page  |2026-01-01 10:00:00|
|2      |add_to_cart|2026-01-01 10:10:00|
|3      |view_page  |2026-01-01 11:00:00|
|4      |view_page  |2026-01-02 12:00:00|
|4      |purchase   |2026-01-02 12:20:00|
|5      |add_to_cart|2026-01-02 13:00:00|
|6      |view_page  |2026-01-03 14:00:00|
|6      |add_to_cart|2026-01-03 14:05:00|
|6      |purchase   |2026-01-03 14:15:00|
+-------+-----------+-------------------+



In [19]:
funnel_df = spark.sql("""
WITH user_steps AS (
    SELECT
        user_id,

        MAX(CASE WHEN event_name = 'view_page' THEN 1 ELSE 0 END) AS viewed_page,
        MAX(CASE WHEN event_name = 'add_to_cart' THEN 1 ELSE 0 END) AS added_to_cart,
        MAX(CASE WHEN event_name = 'purchase' THEN 1 ELSE 0 END) AS purchased

    FROM events
    GROUP BY user_id
)

SELECT
    COUNT(DISTINCT CASE
        WHEN viewed_page = 1
        THEN user_id
    END) AS viewed_users,

    COUNT(DISTINCT CASE
        WHEN viewed_page = 1
         AND added_to_cart = 1
        THEN user_id
    END) AS added_to_cart_users,

    COUNT(DISTINCT CASE
        WHEN viewed_page = 1
         AND added_to_cart = 1
         AND purchased = 1
        THEN user_id
    END) AS purchased_users,

    ROUND(
        COUNT(DISTINCT CASE
            WHEN viewed_page = 1
             AND added_to_cart = 1
            THEN user_id
        END)
        / NULLIF(COUNT(DISTINCT CASE
            WHEN viewed_page = 1
            THEN user_id
        END), 0) * 100,
        2
    ) AS view_to_cart_percent,

    ROUND(
        COUNT(DISTINCT CASE
            WHEN viewed_page = 1
             AND added_to_cart = 1
             AND purchased = 1
            THEN user_id
        END)
        / NULLIF(COUNT(DISTINCT CASE
            WHEN viewed_page = 1
            THEN user_id
        END), 0) * 100,
        2
    ) AS view_to_purchase_percent

FROM user_steps
""")

funnel_df.show()

+------------+-------------------+---------------+--------------------+------------------------+
|viewed_users|added_to_cart_users|purchased_users|view_to_cart_percent|view_to_purchase_percent|
+------------+-------------------+---------------+--------------------+------------------------+
|           5|                  3|              2|                60.0|                    40.0|
+------------+-------------------+---------------+--------------------+------------------------+



In [20]:
ordered_funnel_df = spark.sql("""
WITH step_times AS (
    SELECT
        user_id,

        MIN(CASE WHEN event_name = 'view_page' THEN event_time END) AS first_view_time,
        MIN(CASE WHEN event_name = 'add_to_cart' THEN event_time END) AS first_cart_time,
        MIN(CASE WHEN event_name = 'purchase' THEN event_time END) AS first_purchase_time

    FROM events
    GROUP BY user_id
),

funnel_flags AS (
    SELECT
        user_id,

        CASE
            WHEN first_view_time IS NOT NULL
            THEN 1 ELSE 0
        END AS viewed_page,

        CASE
            WHEN first_view_time IS NOT NULL
             AND first_cart_time IS NOT NULL
             AND first_cart_time >= first_view_time
            THEN 1 ELSE 0
        END AS added_to_cart_after_view,

        CASE
            WHEN first_view_time IS NOT NULL
             AND first_cart_time IS NOT NULL
             AND first_purchase_time IS NOT NULL
             AND first_cart_time >= first_view_time
             AND first_purchase_time >= first_cart_time
            THEN 1 ELSE 0
        END AS purchased_after_cart

    FROM step_times
)

SELECT
    SUM(viewed_page) AS viewed_users,
    SUM(added_to_cart_after_view) AS added_to_cart_users,
    SUM(purchased_after_cart) AS purchased_users,

    ROUND(
        SUM(added_to_cart_after_view) / NULLIF(SUM(viewed_page), 0) * 100,
        2
    ) AS view_to_cart_percent,

    ROUND(
        SUM(purchased_after_cart) / NULLIF(SUM(viewed_page), 0) * 100,
        2
    ) AS view_to_purchase_percent,

    ROUND(
        SUM(purchased_after_cart) / NULLIF(SUM(added_to_cart_after_view), 0) * 100,
        2
    ) AS cart_to_purchase_percent

FROM funnel_flags
""")

ordered_funnel_df.show()

+------------+-------------------+---------------+--------------------+------------------------+------------------------+
|viewed_users|added_to_cart_users|purchased_users|view_to_cart_percent|view_to_purchase_percent|cart_to_purchase_percent|
+------------+-------------------+---------------+--------------------+------------------------+------------------------+
|           5|                  3|              2|                60.0|                    40.0|                   66.67|
+------------+-------------------+---------------+--------------------+------------------------+------------------------+



In [21]:
# Day-1 / Day-7 / Day-30 retention for new users.


In [22]:
retention_df = spark.sql("""
SELECT
    DATE_FORMAT(u.signup_date, 'yyyy-MM') AS signup_month,
    COUNT(DISTINCT u.user_id) AS new_users,
    COUNT(DISTINCT CASE
        WHEN a.activity_date = DATE_ADD(u.signup_date, 1)
        THEN u.user_id
    END) AS day_1_retained_users,
    COUNT(DISTINCT CASE
        WHEN a.activity_date = DATE_ADD(u.signup_date, 7)
        THEN u.user_id
    END) AS day_7_retained_users,
    COUNT(DISTINCT CASE
        WHEN a.activity_date = DATE_ADD(u.signup_date, 30)
        THEN u.user_id
    END) AS day_30_retained_users,
    ROUND(
        COUNT(DISTINCT CASE
            WHEN a.activity_date = DATE_ADD(u.signup_date, 1)
            THEN u.user_id
        END) / COUNT(DISTINCT u.user_id) * 100,
        2
    ) AS day_1_retention_percent,
    ROUND(
        COUNT(DISTINCT CASE
            WHEN a.activity_date = DATE_ADD(u.signup_date, 7)
            THEN u.user_id
        END) / COUNT(DISTINCT u.user_id) * 100,
        2
    ) AS day_7_retention_percent,
    ROUND(
        COUNT(DISTINCT CASE
            WHEN a.activity_date = DATE_ADD(u.signup_date, 30)
            THEN u.user_id
        END) / COUNT(DISTINCT u.user_id) * 100,
        2
    ) AS day_30_retention_percent
FROM users u
LEFT JOIN user_activity a
    ON u.user_id = a.user_id

GROUP BY
    DATE_FORMAT(u.signup_date, 'yyyy-MM')

ORDER BY
    signup_month
""")

retention_df.show()

+------------+---------+--------------------+--------------------+---------------------+-----------------------+-----------------------+------------------------+
|signup_month|new_users|day_1_retained_users|day_7_retained_users|day_30_retained_users|day_1_retention_percent|day_7_retention_percent|day_30_retention_percent|
+------------+---------+--------------------+--------------------+---------------------+-----------------------+-----------------------+------------------------+
|     2026-01|        3|                   1|                   0|                    0|                  33.33|                    0.0|                     0.0|
|     2026-02|        2|                   0|                   0|                    0|                    0.0|                    0.0|                     0.0|
|     2026-03|        1|                   0|                   0|                    0|                    0.0|                    0.0|                     0.0|
+------------+---------+----